# Generate image using scatch image & text prompt

In [11]:
# text encoder from Stable Diffusion
from diffusers import StableDiffusionPipeline
from PIL import Image
import torch

model_id = "sd-legacy/stable-diffusion-v1-5"
sd_pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
sd_pipe = sd_pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


In [20]:
init_image = Image.open('./sub-01/sub-01-sketch.PNG')
prompt = "Photorealistic image of a melancholic dawn cityscape. A solitary woman with disheveled hair stands alone on a stark industrial bridge, photographed from behind, her lonely silhouette small against the harsh rising sun. High-resolution 8K, dark mood photography. The scene captures a cold, stark morning before 6 AM, with a looming suspension bridge casting heavy shadows. Scattered people in the distance emphasize the protagonist's isolation. The morning sun breaks through ominous storm clouds, creating a dramatic but muted orange gradient across a threatening sky. Urban decay elements include weathered bridge architecture, industrial cityscape, and cold structural details. Natural elements show bitter wind violently tousling the woman's hair, heavy mist creating a oppressive atmosphere. Desaturated color palette with selective emphasis on harsh orange sunlight. Cinematic neo-noir style, emphasizing solitude and urban alienation."
image = sd_pipe(prompt=prompt, image=init_image, strength=0.0, guidance_scale=7.5).images[0]
image.save("sub-01-output-1.png")

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ["the distance emphasize the protagonist's isolation. the morning sun breaks through ominous storm clouds, creating a dramatic but muted orange gradient across a threatening sky. urban decay elements include weathered bridge architecture, industrial cityscape, and cold structural details. natural elements show bitter wind violently tousling the woman's hair, heavy mist creating a oppressive atmosphere. desaturated color palette with selective emphasis on harsh orange sunlight. cinematic neo - noir style, emphasizing solitude and urban alienation."]


  0%|          | 0/50 [00:00<?, ?it/s]

# Generate affect image frames

In [78]:
emphasis_words = [
    "incredibly", "astonishingly", "extremely", "wildly", "intensely",
    "passionately", "overwhelmingly", "remarkably", "tremendously",
    "profoundly", "vividly", "electrifyingly", "stunningly",
    "dramatically", "heartfelt", "deeply", "fiercely",
    "exuberantly", "ecstatically", "captivatingly"
]
pos_words = [
    "happiness", "joy", "love", "delight", "inspiration", 
    "wonder", "beauty", "serenity", "contentment", "euphoria", 
    "exhilaration", "radiance", "cheerfulness", "optimism", "upliftment", 
    "friendship", "warmth", "tranquility", "blessing", "appreciation"
]
neg_words = [
    "sad", "angry", "disappointed", "frustrated", "anxious",
    "worried", "gloomy", "hopeless", "lonely", "fearful",
    "depressed", "overwhelmed", "irritated", "isolated", "heavy-hearted",
    "heartbroken", "distressed", "grief-stricken", "mournful", "frightened"
]
print(len(emphasis_words), len(pos_words), len(neg_words))

20 20 20


In [83]:
import requests
import torch
from PIL import Image
from io import BytesIO

from diffusers import StableDiffusionImg2ImgPipeline

device = "cuda"
model_id_or_path = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id_or_path, torch_dtype=torch.float16)
pipe = pipe.to(device)

init_image = Image.open('./sub-02-sketch.png').convert("RGB")
init_image = init_image.resize((768, 512))
prompt = "A realistic photo inside view of a city bus, from the back seat of the bus with multiple bus chairs. Insane sunshine in front. The hot and intense sunlight streams in through the front window of the bus. There is a bus driver sitting on the bus chair. The front windshield is visible, showing only sky and sun due to the bus climbing a very steep hill. Late afternoon, around 5 PM, with the setting sun insanely casting a deep orange light through the front windshield. Cinematic lighting, dramatic angle, urban photography, realistic style, high contrast, golden hour, lens flare, interior vehicle shot, public transportation aesthetics. Ultra detailed, realistic, insanely beautiful."

affect_prompts = []
for t in range(len(affect_ratio)):
    affect_prompt = "The moods of this photo are "
    print(t, np.rint(affect_ratio[t]*10))
    
    #positive
    for i in range(int(np.rint(affect_ratio[t][1]*10))):
        emp_r = np.random.randint(20)
        pos_r = np.random.randint(20)    
        affect_prompt += emphasis_words[emp_r]+" "+pos_words[pos_r]
        if i != int(np.rint(affect_ratio[t][1]*10))-1:
            affect_prompt += " and "

    #negative
    for i in range(int(np.rint(affect_ratio[t][2]*10))):
        emp_r = np.random.randint(20)
        neg_r = np.random.randint(20)
        affect_prompt += emphasis_words[emp_r]+" "+neg_words[neg_r]
        if i != int(np.rint(affect_ratio[t][2]*10))-1:
            affect_prompt += " and "

    affect_prompt += "."
    print(affect_prompt)
    affect_prompts.append(affect_prompt)

    
images = pipe(prompt=prompt, image=init_image, strength=0.75, guidance_scale=7.5).images
images[0].save("sub-02_affect_non.png")

for i in range(len(affect_ratio)):
    all_prompt = prompt+" "+affect_prompts[i]
    print(all_prompt)

    images = pipe(prompt=all_prompt, image=init_image, strength=0.75, guidance_scale=7.5).images
    images[0].save(f"sub-02_affect_{(i+1)*10000}.png")

2024-10-04 01:46:32,237 - DEBUG - https://huggingface.co:443 "GET /api/models/runwayml/stable-diffusion-v1-5 HTTP/1.1" 307 90
2024-10-04 01:46:32,334 - DEBUG - https://huggingface.co:443 "GET /api/models/stable-diffusion-v1-5/stable-diffusion-v1-5 HTTP/1.1" 200 2843
2024-10-04 01:46:32,425 - DEBUG - https://huggingface.co:443 "HEAD /runwayml/stable-diffusion-v1-5/resolve/main/model_index.json HTTP/1.1" 307 0
2024-10-04 01:46:32,522 - DEBUG - https://huggingface.co:443 "HEAD /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/model_index.json HTTP/1.1" 200 0


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
2024-10-04 01:46:36,501 - DEBUG - STREAM b'IHDR' 16 13
2024-10-04 01:46:36,502 - DEBUG - STREAM b'iCCP' 41 373
2024-10-04 01:46:36,502 - DEBUG - iCCP profile name b'kCGColorSpaceDisplayP3'
2024-10-04 01:46:36,503 - DEBUG - Compression method 0
2024-10-04 01:46:36,503 - DEBUG - STREAM b'eXIf' 426 120
2024-10-04 01:46:36,504 - DEBUG - STREAM b'pHYs' 558 9
2024-10-04 01:46:36,504 - DEBUG - STREAM b'iTXt' 579 914
2024-10-04 01:46:36,505 - DEBUG - STREAM b'IDAT' 1505 16384
Token indices sequence length is longer than the specified maximum sequence length for this model (135 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['afternoon, around 5 pm, with the setting sun insanely casting a deep orange light through the front wind

0 [7. 0. 3.]
The moods of this photo are extremely fearful and overwhelmingly disappointed and extremely isolated.
1 [6. 3. 1.]
The moods of this photo are incredibly wonder and incredibly exhilaration and deeply appreciationstunningly worried.
2 [5. 3. 2.]
The moods of this photo are tremendously radiance and astonishingly upliftment and captivatingly beautyremarkably mournful and passionately grief-stricken.
3 [8. 0. 2.]
The moods of this photo are dramatically depressed and intensely overwhelmed.
4 [8. 2. 0.]
The moods of this photo are captivatingly inspiration and tremendously wonder.


  0%|          | 0/37 [00:00<?, ?it/s]

...

A realistic photo inside view of a city bus, from the back seat of the bus with multiple bus chairs. Insane sunshine in front. The hot and intense sunlight streams in through the front window of the bus. There is a bus driver sitting on the bus chair. The front windshield is visible, showing only sky and sun due to the bus climbing a very steep hill. Late afternoon, around 5 PM, with the setting sun insanely casting a deep orange light through the front windshield. Cinematic lighting, dramatic angle, urban photography, realistic style, high contrast, golden hour, lens flare, interior vehicle shot, public transportation aesthetics. Ultra detailed, realistic, insanely beautiful. The moods of this photo are dramatically depressed and intensely overwhelmed.


  0%|          | 0/37 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['afternoon, around 5 pm, with the setting sun insanely casting a deep orange light through the front windshield. cinematic lighting, dramatic angle, urban photography, realistic style, high contrast, golden hour, lens flare, interior vehicle shot, public transportation aesthetics. ultra detailed, realistic, insanely beautiful. the moods of this photo are captivatingly inspiration and tremendously wonder.']


A realistic photo inside view of a city bus, from the back seat of the bus with multiple bus chairs. Insane sunshine in front. The hot and intense sunlight streams in through the front window of the bus. There is a bus driver sitting on the bus chair. The front windshield is visible, showing only sky and sun due to the bus climbing a very steep hill. Late afternoon, around 5 PM, with the setting sun insanely casting a deep orange light through the front windshield. Cinematic lighting, dramatic angle, urban photography, realistic style, high contrast, golden hour, lens flare, interior vehicle shot, public transportation aesthetics. Ultra detailed, realistic, insanely beautiful. The moods of this photo are captivatingly inspiration and tremendously wonder.


  0%|          | 0/37 [00:00<?, ?it/s]

# Generate Video from Image frames

In [84]:
# Image2Video
import time
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from DeepCache.svd.pipeline_stable_video_diffusion import StableVideoDiffusionPipeline as DeepCacheStableVideoDiffusionPipeline

import logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

frames_lst = []
file_name = f"sub-02_affect"
for i in range(5):
    # Load the conditioning image
    image = load_image(f"./sub-02_affect_{10000*(i+1)}.png")
    # image = image.resize((1024, 576))

    pipe = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt", torch_dtype=torch.float16, variant="fp16"
    )
    pipe.enable_model_cpu_offload()

    generator = torch.manual_seed(84)
    logging.info("Running baseline...")
    start_time = time.time()
    frames = pipe(
        image, 
        decode_chunk_size=8, generator=generator,
    ).frames[0]
    for j in range(len(frames)):
        frames_lst.append(frames[j])
    origin_use_time = time.time() - start_time

export_to_video(frames_lst, "{}.mp4".format(file_name), fps=7)
logging.info("Origin: {:.2f} seconds".format(origin_use_time))

del pipe
torch.cuda.empty_cache()

2024-10-04 01:47:23,494 - DEBUG - STREAM b'IHDR' 16 13
2024-10-04 01:47:23,494 - DEBUG - STREAM b'IDAT' 41 65536
2024-10-04 01:47:23,606 - DEBUG - https://huggingface.co:443 "GET /api/models/stabilityai/stable-video-diffusion-img2vid-xt HTTP/1.1" 200 4215
2024-10-04 01:47:23,704 - DEBUG - https://huggingface.co:443 "HEAD /stabilityai/stable-video-diffusion-img2vid-xt/resolve/main/model_index.json HTTP/1.1" 200 0


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

2024-10-04 01:47:25,125 - INFO - Running baseline...


  0%|          | 0/25 [00:00<?, ?it/s]

2024-10-04 01:48:19,916 - DEBUG - STREAM b'IHDR' 16 13
2024-10-04 01:48:19,917 - DEBUG - STREAM b'IDAT' 41 65536
2024-10-04 01:48:20,031 - DEBUG - https://huggingface.co:443 "GET /api/models/stabilityai/stable-video-diffusion-img2vid-xt HTTP/1.1" 200 4215
2024-10-04 01:48:20,124 - DEBUG - https://huggingface.co:443 "HEAD /stabilityai/stable-video-diffusion-img2vid-xt/resolve/main/model_index.json HTTP/1.1" 200 0


...

2024-10-04 01:51:12,345 - DEBUG - STREAM b'IHDR' 16 13
2024-10-04 01:51:12,346 - DEBUG - STREAM b'IDAT' 41 65536
2024-10-04 01:51:12,451 - DEBUG - https://huggingface.co:443 "GET /api/models/stabilityai/stable-video-diffusion-img2vid-xt HTTP/1.1" 200 4215
2024-10-04 01:51:12,560 - DEBUG - https://huggingface.co:443 "HEAD /stabilityai/stable-video-diffusion-img2vid-xt/resolve/main/model_index.json HTTP/1.1" 200 0


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

2024-10-04 01:51:13,929 - INFO - Running baseline...


  0%|          | 0/25 [00:00<?, ?it/s]

2024-10-04 01:52:09,980 - INFO - Origin: 55.43 seconds
